# L14 · 多股选股（Stock Selection）

**预计时长**：90 min | **难度**：⭐⭐⭐⭐ | **前置**：L04 panel 对齐、L08 PE 估值

## 本节目标
1. 从「单股回测」跃迁到「多股选股 + 组合」
2. 三大经典选股因子：价值（PE）/ 动量（Momentum）/ 波动率（Volatility）
3. 多因子 rank 融合：每个因子打分 → 综合排名 → top-N 选股
4. 分位回测（quintile backtest）：把股票按因子分 5 组，看 top 组 vs bottom 组

## 第 1 段：金融概念

### 主动选股 vs 被动持有
- **被动**（买入持有 ETF）：拿到市场平均 β 收益
- **主动选股**：试图获取 α 超额收益——但这非常难

### 三大经典因子（Fama-French + 行为金融）
| 因子 | 含义 | 历史 long-term 表现 |
|------|------|-------------------|
| **价值（PE/PB）** | 低估值股票便宜 | 价值溢价，但 2010-2020 美股失效 |
| **动量（Momentum）** | 过去 6-12 月涨得多的继续涨 | 持续有效，但崩溃剧烈 |
| **低波动（LowVol）** | 波动率低的股票长期更优 | 低波动异象，反 CAPM 直觉 |
| 质量（Quality） | ROE 高、负债低 | 长期有效 |
| 规模（Size） | 市值小 → 效率高 | A 股小盘股效应显著 |

### 多因子 rank 融合流程
```
1. 每个股票每个因子 → 计算因子值 → 在全市场内 rank (0~1)
2. 多因子加权：composite = 0.4*PE_rank + 0.3*MOM_rank + 0.3*LowVol_rank
3. 选 composite 最高的 top-N（如 top 20%）
```

### A 股特性
- 小盘股效应非常强（中证 1000 / 中证 2000 长期跑赢沪深 300）
- 题材/概念驱动，价值因子在牛市失效、熊市有效
- 财务造假风险 → 质量因子（剔除高应收/低现金流）尤其重要

In [ ]:
import sys
from pathlib import Path

# 自动定位 phase1_foundation（共享 _data/_style）+ project root
_cwd = Path.cwd()
_p1 = _cwd if (_cwd / '_data.py').exists() else (
    _cwd / 'learning' / 'phase1_foundation'
    if (_cwd / 'learning' / 'phase1_foundation' / '_data.py').exists()
    else _cwd.parent / 'phase1_foundation'
)
sys.path.insert(0, str(_p1))
_proj = _p1.parent.parent if _p1.name == 'phase1_foundation' else _p1
if (_proj / 'qtrader' / '__init__.py').exists():
    sys.path.insert(0, str(_proj))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import _style
from _data import get_stock_data
from qtrader import (
    BacktestEngine, CostModel,
    DualMAStrategy, TurtleStrategy, GridStrategy,
    print_comparison_table,
)

## 第 2 段：构建多股 panel

复用 Phase 1 L04 的多股对齐能力。

In [ ]:
# 5 只代表性股票跨行业
codes_names = [
    ('002594', '比亚迪'),    # 汽车
    ('600519', '贵州茅台'),  # 食品饮料
    ('002624', '完美世界'),  # 传媒
    ('300750', '宁德时代'),  # 电池
    ('000001', '平安银行'),  # 银行
]
codes = [c for c, _ in codes_names]
names = [n for _, n in codes_names]

frames = {}
for code in codes:
    df = get_stock_data(code).set_index('date')
    frames[code] = df['close']
panel = pd.DataFrame(frames).sort_index().ffill()
print(f"panel shape: {panel.shape}")
print(panel.tail(3))

## 第 3 段：计算三大因子

In [ ]:
rets_panel = panel.pct_change().dropna()

# 因子 1: 动量（过去 6 个月累计收益）
momentum = panel.pct_change(126).iloc[-1]  # 最近一日的 6 月动量
print("动量（近 6 月涨幅）:")
print(momentum.sort_values(ascending=False).round(3))

# 因子 2: 波动率（过去 1 年日波动率年化）
vol = rets_panel.std() * np.sqrt(252)
print("\n年化波动率:")
print(vol.sort_values().round(3))

# 因子 3: 价值（PE — 这里用代理指标：价格分位）
# 真实 PE 需要财务数据，这里用「当前价在过去 1 年的分位」作为代理
pe_proxy = panel.apply(lambda s: (s.iloc[-1] - s.min()) / (s.max() - s.min()))
print("\n价格分位（0 = 最低，1 = 最高）:")
print(pe_proxy.sort_values().round(3))

## 第 4 段：多因子 rank 融合

关键：每个因子先 rank → 再加权，避免量纲不一致。

In [ ]:
factors = pd.DataFrame({
    'momentum': momentum,
    'low_vol': -vol,      # 负号：波动低 = 好
    'value': -pe_proxy,   # 负号：价格分位低 = 便宜 = 好
})

# 在每列内 rank（0~1）
ranked = factors.rank(pct=True)
print("各因子分位排名（越大越好）:")
print(ranked.round(3))

# 加权融合
weights = {'momentum': 0.4, 'low_vol': 0.3, 'value': 0.3}
composite = (
    ranked['momentum'] * weights['momentum']
    + ranked['low_vol'] * weights['low_vol']
    + ranked['value'] * weights['value']
)

print("\n综合排名（降序，前 2 名 = 推荐买入）:")
top_n = 2
top_codes = composite.sort_values(ascending=False).head(top_n)
print(top_codes.round(3))

# 加上股票名
code_to_name = dict(codes_names)
print("\n推荐组合:")
for code, score in top_codes.items():
    print(f"  {code} {code_to_name[code]:<6} 综合分: {score:.3f}")

## 第 5 段：分位回测（Quintile Backtest）

把股票按因子分 5 组，看 top 组是否长期跑赢 bottom 组。
由于这里只有 5 只股票，分组意义有限——但流程对真实 300+ 股票池同样适用。

In [ ]:
# 简化：按动量因子排序，top 组 vs bottom 组
sorted_by_mom = momentum.sort_values(ascending=False)
top_group = sorted_by_mom.head(2).index.tolist()
bottom_group = sorted_by_mom.tail(2).index.tolist()

# 等权组合净值
top_nav = (1 + rets_panel[top_group].mean(axis=1)).cumprod()
bottom_nav = (1 + rets_panel[bottom_group].mean(axis=1)).cumprod()

fig, ax = plt.subplots(figsize=(13, 5))
top_nav.plot(ax=ax, label=f'Top 动量 ({", ".join(top_group)})', linewidth=2)
bottom_nav.plot(ax=ax, label=f'Bottom 动量 ({", ".join(bottom_group)})', linestyle='--')
ax.set_title('动量因子分位回测：Top vs Bottom')
ax.legend()
plt.tight_layout(); plt.show()

print(f"Top 组年化:    {(top_nav.iloc[-1] ** (252/len(top_nav)) - 1) * 100:.2f}%")
print(f"Bottom 组年化: {(bottom_nav.iloc[-1] ** (252/len(bottom_nav)) - 1) * 100:.2f}%")
print(f"多空收益:      {(top_nav.iloc[-1] / bottom_nav.iloc[-1] - 1) * 100:.2f}%")

## 第 6 段：本节要点

1. **选股 = 从 β 到 α 的跃迁**：被动持有拿 β，主动选股追求 α
2. **三大因子**：价值、动量、低波动，每个都有学术与实战依据
3. **多因子融合**：先 rank 再加权，避免量纲问题
4. **分位回测**：是评估因子有效性的标准方法
5. **A 股特性**：小盘效应强、题材驱动、财务造假风险——质量因子必备

### 🔮 下节 L15：组合构建
选出 top-N 股票后，每只配置多少仓位？等权 / 反向波动加权 / Markowitz。